<a href="https://colab.research.google.com/github/sinbbang/L-IntelligentSystem/blob/main/%EC%A7%80%EB%8A%A5%EC%8B%9C%EC%8A%A4%ED%85%9C_07-6_Budget_Guard_%EC%83%81%ED%83%9C%EB%88%84%EC%A0%81_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 실습 6 — Budget Guard: 상태 누적과 실행 제한

**학습 목표**
1. `steps`, `tool_calls`, `tokens`를 Python 코드가 직접 누적한다.
2. 예산 한도를 넘기기 **전에** 다음 행동을 차단하는 방법을 이해한다.
3. 제한에 도달한 이유(`max_steps`, `max_tool_calls`, `max_tokens`)를 로그로 확인한다.

> 이 실습은 별도의 LLM API 키 없이 Google Colab에서 실행됩니다. `token_usage`는 아래 시뮬레이션에서 지정한 **가상 토큰 사용량**이며 실제 LLM 사용량이 아닙니다. 실제 서비스에서는 API가 제공하는 usage 메타데이터 등을 사용해야 합니다.

## 1. Budget 클래스 만들기


In [1]:
class Budget:
    def __init__(self, max_steps=8, max_tool_calls=5, max_tokens=20_000):
        # 사용할 수 있는 최대 예산
        self.max_steps = max_steps
        self.max_tool_calls = max_tool_calls
        self.max_tokens = max_tokens

        # 실행 중 현재까지 사용한 양
        self.steps = 0
        self.tool_calls = 0
        self.tokens = 0

    def add_step(self, token_usage, used_tool):
        """완료된 한 단계의 실제 사용량을 누적합니다."""
        if not isinstance(token_usage, int) or token_usage < 0:
            raise ValueError("token_usage는 0 이상의 정수여야 합니다.")
        if not isinstance(used_tool, bool):
            raise TypeError("used_tool은 True 또는 False여야 합니다.")

        self.steps += 1
        self.tokens += token_usage
        if used_tool:
            self.tool_calls += 1

    def snapshot(self):
        """현재 누적 사용량을 사전(dict)으로 반환합니다."""
        return {"steps": self.steps,
                "tool_calls": self.tool_calls,
                "tokens": self.tokens}

budget = Budget()
budget.add_step(token_usage=120, used_tool=True)
budget.add_step(token_usage=80, used_tool=False)
print("현재 누적값:", budget.snapshot())
assert budget.snapshot() == {"steps": 2, "tool_calls": 1, "tokens": 200}

현재 누적값: {'steps': 2, 'tool_calls': 1, 'tokens': 200}


## 2. 예산 초과를 사전에 확인하기

`add_step()`은 슬라이드처럼 **사용량을 누적하는 역할**을 합니다. 그러나 누적만 한다면 예산 초과를 막을 수 없습니다.

따라서 새 동작을 시작하기 전에 예상 소비량에 따라 허용 여부를 판정하는 별도 함수를 만듭니다.

- `steps + 1 > max_steps`: 다음 단계 금지
- `tool_calls + 1 > max_tool_calls`: 다음 도구 호출 금지
- `tokens + 예정 토큰 > max_tokens`: 다음 작업 금지

**주의:** 실제 LLM 호출의 토큰 수는 호출 전에 정확히 알 수 없습니다. 아래의 `planned_tokens`는 교육용으로 미리 정한 소비량입니다. 실제 운영에서는 사전 추정/요청별 한도와 호출 후 실제 사용량 정산이 필요합니다.

In [2]:
def check_budget(budget, planned_tokens, used_tool):
    """다음 단계가 예산 안에서 실행 가능한지 확인합니다."""
    if not isinstance(planned_tokens, int) or planned_tokens < 0:
        raise ValueError("planned_tokens는 0 이상의 정수여야 합니다.")
    if not isinstance(used_tool, bool):
        raise TypeError("used_tool은 True 또는 False여야 합니다.")

    if budget.steps + 1 > budget.max_steps:
        return False, "max_steps"
    if used_tool and budget.tool_calls + 1 > budget.max_tool_calls:
        return False, "max_tool_calls"
    if budget.tokens + planned_tokens > budget.max_tokens:
        return False, "max_tokens"
    return True, "ok"

example = Budget(max_steps=2, max_tool_calls=1, max_tokens=300)
print(check_budget(example, planned_tokens=100, used_tool=True))  # (True, 'ok')
example.add_step(100, True)
print(check_budget(example, planned_tokens=100, used_tool=True))  # 도구 호출 한도 초과

(True, 'ok')
(False, 'max_tool_calls')


## 3. 간단한 도구 및 Harness 시뮬레이터
모의 Policy가 내놓은 행동을 순서대로 처리합니다. `search`는 미리 정의된 문자열을 돌려주는 교육용 도구입니다.

In [3]:
def search_course(query):
    """강의 정보 검색을 흉내 내는 도구입니다."""
    if "평가" in query:
        return "평가 비율: 과제 20%, 중간 30%, 기말 30%, 출석 20%"
    if "AI" in query:
        return "AI 사용 정책: 활용 가능 범위 준수, 출처 명시"
    return "검색 결과 없음"

TOOLS = {"search": search_course}

def run_with_budget(actions, budget, verbose=True):
    """동작 목록을 순서대로 실행하며 예산과 로그를 관리합니다.

    각 action: {"tool": 도구명 또는 finish, "args": {...}, "tokens": 교육용 정수}
    """
    log = []
    answer = None
    status = "actions_exhausted"

    for action in actions:
        tool_name = action.get("tool")
        args = action.get("args", {})
        token_usage = action.get("tokens", 0)

        # finish는 의사결정 단계이지만 외부 도구 호출은 아닙니다.
        used_tool = tool_name != "finish"
        allowed, reason = check_budget(budget, token_usage, used_tool)
        if not allowed:
            status = reason
            log.append({"event": "blocked", "reason": reason,
                        "next_tool": tool_name, **budget.snapshot()})
            break

        # 허용된 도구인지 확인합니다. 차단된 행동은 사용량에 반영하지 않습니다.
        if tool_name != "finish" and tool_name not in TOOLS:
            status = "invalid_tool"
            log.append({"event": "blocked", "reason": status,
                        "next_tool": tool_name, **budget.snapshot()})
            break

        if not isinstance(args, dict):
            status = "invalid_args"
            log.append({"event": "blocked", "reason": status,
                        "next_tool": tool_name, **budget.snapshot()})
            break

        if tool_name == "finish":
            answer = action.get("answer", "작업 완료")
            result = answer
            status = "completed"
        else:
            try:
                result = TOOLS[tool_name](**args)
            except (TypeError, ValueError) as exc:
                status = "tool_error"
                log.append({"event": "blocked", "reason": status,
                            "detail": str(exc), **budget.snapshot()})
                break

        # 한 단계가 끝나면 코드가 직접 사용량을 누적합니다.
        budget.add_step(token_usage=token_usage, used_tool=used_tool)
        log.append({"event": "executed", "tool": tool_name,
                    "result": result, **budget.snapshot()})
        if status == "completed":
            break

    if verbose:
        print("종료 상태:", status)
        print("최종 답변:", answer)
        for row in log:
            print(row)
    return answer, log, status


## 4. 정상 실행: 예산 안에서 완료하기

In [4]:
ACTIONS_OK = [
    {"tool": "search", "args": {"query": "지능시스템 평가"}, "tokens": 120},
    {"tool": "search", "args": {"query": "지능시스템 AI 정책"}, "tokens": 160},
    {"tool": "finish", "answer": "평가 정보와 AI 정책을 확인했습니다.", "tokens": 40},
]

ok_budget = Budget(max_steps=4, max_tool_calls=2, max_tokens=500)
answer, log, status = run_with_budget(ACTIONS_OK, ok_budget)
assert status == "completed"
assert ok_budget.snapshot() == {"steps": 3, "tool_calls": 2, "tokens": 320}

종료 상태: completed
최종 답변: 평가 정보와 AI 정책을 확인했습니다.
{'event': 'executed', 'tool': 'search', 'result': '평가 비율: 과제 20%, 중간 30%, 기말 30%, 출석 20%', 'steps': 1, 'tool_calls': 1, 'tokens': 120}
{'event': 'executed', 'tool': 'search', 'result': 'AI 사용 정책: 활용 가능 범위 준수, 출처 명시', 'steps': 2, 'tool_calls': 2, 'tokens': 280}
{'event': 'executed', 'tool': 'finish', 'result': '평가 정보와 AI 정책을 확인했습니다.', 'steps': 3, 'tool_calls': 2, 'tokens': 320}


## 5. 예산별 차단 상황 실험하기
동일한 행동 목록을 유지하고 한도만 변경합니다. **어느 단계 직전에** 차단되는지 확인하세요.

In [5]:
SCENARIOS = [
    ("단계 수 제한", Budget(max_steps=2, max_tool_calls=5, max_tokens=1000)),
    ("도구 호출 수 제한", Budget(max_steps=5, max_tool_calls=1, max_tokens=1000)),
    ("토큰 예산 제한", Budget(max_steps=5, max_tool_calls=5, max_tokens=200)),
]

for name, budget in SCENARIOS:
    print(f"\n[{name}]")
    _, _, status = run_with_budget(ACTIONS_OK, budget, verbose=False)
    print("종료 상태:", status)
    print("누적 사용량:", budget.snapshot())

assert run_with_budget(ACTIONS_OK, Budget(2, 5, 1000), False)[2] == "max_steps"
assert run_with_budget(ACTIONS_OK, Budget(5, 1, 1000), False)[2] == "max_tool_calls"
assert run_with_budget(ACTIONS_OK, Budget(5, 5, 200), False)[2] == "max_tokens"


[단계 수 제한]
종료 상태: max_steps
누적 사용량: {'steps': 2, 'tool_calls': 2, 'tokens': 280}

[도구 호출 수 제한]
종료 상태: max_tool_calls
누적 사용량: {'steps': 1, 'tool_calls': 1, 'tokens': 120}

[토큰 예산 제한]
종료 상태: max_tokens
누적 사용량: {'steps': 1, 'tool_calls': 1, 'tokens': 120}
